# glcuda Wave 75 - row-major compensated-MMA AV direct gate`n`nThis notebook validates the retained-layout candidate on an actual Tesla T4.`n

In [ ]:
import base64
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import traceback
import urllib.request
import zipfile

BASE_REV = '3bce8dd7b8aaa2765855ab927c611b54981f9241'
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
EMBEDDED = json.loads('[{"path":"glcuda/src/kernels/glcuda_sm75_wave64.ptx","sha256":"ef12c6cfbe495535dffc6f6402fb194aa6f60a3011a333a1a1ff7584c3c7eb1f","base64":"LnZlcnNpb24gNy4wCi50YXJnZXQgc21fNzUKLmFkZHJlc3Nfc2l6ZSA2NAoKLy8gV2F2ZSA2NCBkaWFnbm9zdGljIG9ubHkuIFRoZXNlIGVudHJpZXMgYXJlIG5vdCBsb2FkZWQgYnkgS2VybmVsU2V0LgovLwovLyBJbnB1dCBwcm9iYWJpbGl0aWVzIGFyZSBwYWNrZWQgW2hlYWRzLCBudG9rLCBjYXBhY2l0eV0gd2l0aCBjYXVzYWwtdGFpbAovLyB6ZXJvcy4gVGhlIHJldGFpbmVkIGVudHJ5IHJlYWRzIHJvdy1tYWpvciBWIFtoZWFkcywgY2FwYWNpdHksIDY0XS4gVGhlCi8vIFdhdmUgNzUgZXZvbHZlcyB0aGUgY2FuZGlkYXRlIHRvIHJlYWQgcmV0YWluZWQgcm93LW1ham9yIFYgZGlyZWN0bHkuIFRoZQovLyB3YXJwJ3MgTU1BIGxhbmUgbWFwcGluZyBjb3ZlcnMgYW4gOHg4IEsvTiB0aWxlOyBpdHMgdHdvIEsgdmFsdWVzIGFyZSBvbmUKLy8gNjQtZmxvYXQgcm93IGFwYXJ0IHJhdGhlciB0aGFuIGFkamFjZW50IGluIGEgdHJhbnNwb3NlZCBpbWFnZS4KLy8gQm90aCB3cml0ZSBwYWNrZWQgZjMyIFtoZWFkcywgbnRvaywgNjRdLgovLwovLyBMYXVuY2g6IGdyaWQgKGNlaWwobnRvay8xNiksIGhlYWRzKSwgYmxvY2sgMTI4LgovLyBDYW5kaWRhdGU6IG9uZSB3YXJwIG93bnMgdHdvIGFkamFjZW50IE44IGZyYWdtZW50cyBvZiB0aGUgMTZ4NjQgb3V0cHV0LgoKLnZpc2libGUgLmVudHJ5IGdsX3dhdmU2NF9hdl9zY2FsYXJfZjMyKAogICAgLnBhcmFtIC51NjQgcF9wcm9iLAogICAgLnBhcmFtIC51NjQgcF92LAogICAgLnBhcmFtIC51NjQgcF9vdXQsCiAgICAucGFyYW0gLnUzMiBwX250b2ssCiAgICAucGFyYW0gLnUzMiBwX2NhcGFjaXR5KQp7CiAgICAucmVnIC5wcmVkICVwPDQ+OwogICAgLnJlZyAuYjMyICVyPDE4PjsKICAgIC5yZWcgLmI2NCAlcmQ8MTY+OwogICAgLnJlZyAuZjMyICVmPDU+OwoKICAgIGxkLnBhcmFtLnU2NCAlcmQxLCBbcF9wcm9iXTsKICAgIGxkLnBhcmFtLnU2NCAlcmQyLCBbcF92XTsKICAgIGxkLnBhcmFtLnU2NCAlcmQzLCBbcF9vdXRdOwogICAgbGQucGFyYW0udTMyICVyMSwgW3BfbnRva107CiAgICBsZC5wYXJhbS51MzIgJXIyLCBbcF9jYXBhY2l0eV07CiAgICBjdnRhLnRvLmdsb2JhbC51NjQgJXJkNCwgJXJkMTsKICAgIGN2dGEudG8uZ2xvYmFsLnU2NCAlcmQ1LCAlcmQyOwogICAgY3Z0YS50by5nbG9iYWwudTY0ICVyZDYsICVyZDM7CgogICAgbW92LnUzMiAlcjMsICVjdGFpZC54OwogICAgc2hsLmIzMiAlcjMsICVyMywgNDsKICAgIG1vdi51MzIgJXI0LCAlY3RhaWQueTsKICAgIG1vdi51MzIgJXI1LCAldGlkLng7CiAgICBtb3YudTMyICVyNiwgJXI1OwoKVzY0X1NDQUxBUl9PVVRQVVQ6CiAgICBzZXRwLmdlLnUzMiAlcDEsICVyNiwgMTAyNDsKICAgIEAlcDEgYnJhIFc2NF9TQ0FMQVJfRE9ORTsKICAgIHNoci51MzIgJXI3LCAlcjYsIDY7CiAgICBhbmQuYjMyICVyOCwgJXI2LCA2MzsKICAgIGFkZC51MzIgJXI5LCAlcjMsICVyNzsKICAgIHNldHAuZ2UudTMyICVwMiwgJXI5LCAlcjE7CiAgICBAJXAyIGJyYSBXNjRfU0NBTEFSX05FWFQ7CgogICAgbXVsLmxvLnUzMiAlcjEwLCAlcjQsICVyMTsKICAgIGFkZC51MzIgJXIxMCwgJXIxMCwgJXI5OwogICAgbXVsLmxvLnUzMiAlcjEwLCAlcjEwLCAlcjI7CiAgICBzaGwuYjMyICVyMTAsICVyMTAsIDI7CiAgICBjdnQudTY0LnUzMiAlcmQ3LCAlcjEwOwogICAgYWRkLnU2NCAlcmQ3LCAlcmQ0LCAlcmQ3OwoKICAgIG11bC5sby51MzIgJXIxMSwgJXI0LCAlcjI7CiAgICBzaGwuYjMyICVyMTEsICVyMTEsIDY7CiAgICBhZGQudTMyICVyMTEsICVyMTEsICVyODsKICAgIHNobC5iMzIgJXIxMSwgJXIxMSwgMjsKICAgIGN2dC51NjQudTMyICVyZDgsICVyMTE7CiAgICBhZGQudTY0ICVyZDgsICVyZDUsICVyZDg7CgogICAgbW92LmYzMiAlZjEsIDBmMDAwMDAwMDA7CiAgICBtb3YudTMyICVyMTIsIDA7Clc2NF9TQ0FMQVJfSzoKICAgIHNldHAuZ2UudTMyICVwMywgJXIxMiwgJXIyOwogICAgQCVwMyBicmEgVzY0X1NDQUxBUl9TVE9SRTsKICAgIGxkLmdsb2JhbC5mMzIgJWYyLCBbJXJkN107CiAgICBsZC5nbG9iYWwuZjMyICVmMywgWyVyZDhdOwogICAgZm1hLnJuLmYzMiAlZjEsICVmMiwgJWYzLCAlZjE7CiAgICBhZGQudTY0ICVyZDcsICVyZDcsIDQ7CiAgICBhZGQudTY0ICVyZDgsICVyZDgsIDI1NjsKICAgIGFkZC51MzIgJXIxMiwgJXIxMiwgMTsKICAgIGJyYSBXNjRfU0NBTEFSX0s7CgpXNjRfU0NBTEFSX1NUT1JFOgogICAgbXVsLmxvLnUzMiAlcjEzLCAlcjQsICVyMTsKICAgIGFkZC51MzIgJXIxMywgJXIxMywgJXI5OwogICAgc2hsLmIzMiAlcjEzLCAlcjEzLCA2OwogICAgYWRkLnUzMiAlcjEzLCAlcjEzLCAlcjg7CiAgICBzaGwuYjMyICVyMTMsICVyMTMsIDI7CiAgICBjdnQudTY0LnUzMiAlcmQ5LCAlcjEzOwogICAgYWRkLnU2NCAlcmQ5LCAlcmQ2LCAlcmQ5OwogICAgc3QuZ2xvYmFsLmYzMiBbJXJkOV0sICVmMTsKClc2NF9TQ0FMQVJfTkVYVDoKICAgIGFkZC51MzIgJXI2LCAlcjYsIDEyODsKICAgIGJyYSBXNjRfU0NBTEFSX09VVFBVVDsKVzY0X1NDQUxBUl9ET05FOgogICAgcmV0Owp9CgoudmlzaWJsZSAuZW50cnkgZ2xfd2F2ZTc1X2F2X21tYTRfcm93X2YzMigKICAgIC5wYXJhbSAudTY0IHBfcHJvYiwKICAgIC5wYXJhbSAudTY0IHBfdiwKICAgIC5wYXJhbSAudTY0IHBfb3V0LAogICAgLnBhcmFtIC51MzIgcF9udG9rLAogICAgLnBhcmFtIC51MzIgcF9jYXBhY2l0eSkKewogICAgLnJlZyAucHJlZCAlcDw2PjsKICAgIC5yZWcgLmIxNiAlaDwxNj47CiAgICAucmVnIC5iMzIgJXI8MzI+OwogICAgLnJlZyAuYjMyICVhX2hpMCwgJWFfaGkxLCAlYV9sbzAsICVhX2xvMTsKICAgIC5yZWcgLmIzMiAlYjBfaGksICViMF9sbywgJWIxX2hpLCAlYjFfbG87CiAgICAucmVnIC5iNjQgJXJkPDIwPjsKICAgIC5yZWcgLmYzMiAlZjwyMD47CiAgICAucmVnIC5mMzIgJWM8OD47CgogICAgbGQucGFyYW0udTY0ICVyZDEsIFtwX3Byb2JdOwogICAgbGQucGFyYW0udTY0ICVyZDIsIFtwX3ZdOwogICAgbGQucGFyYW0udTY0ICVyZDMsIFtwX291dF07CiAgICBsZC5wYXJhbS51MzIgJXIxLCBbcF9udG9rXTsKICAgIGxkLnBhcmFtLnUzMiAlcjIsIFtwX2NhcGFjaXR5XTsKICAgIGN2dGEudG8uZ2xvYmFsLnU2NCAlcmQ0LCAlcmQxOwogICAgY3Z0YS50by5nbG9iYWwudTY0ICVyZDUsICVyZDI7CiAgICBjdnRhLnRvLmdsb2JhbC51NjQgJXJkNiwgJXJkMzsKCiAgICBtb3YudTMyICVyMywgJXRpZC54OwogICAgc2hyLnUzMiAlcjQsICVyMywgNTsKICAgIGFuZC5iMzIgJXI1LCAlcjMsIDMxOwogICAgc2hyLnUzMiAlcjYsICVyNSwgMjsKICAgIGFuZC5iMzIgJXI3LCAlcjUsIDM7CiAgICBtb3YudTMyICVyOCwgJWN0YWlkLng7CiAgICBzaGwuYjMyICVyOCwgJXI4LCA0OwogICAgbW92LnUzMiAlcjksICVjdGFpZC55OwogICAgc2hsLmIzMiAlcjEwLCAlcjQsIDQ7CgogICAgbW92LmYzMiAlYzAsIDBmMDAwMDAwMDA7CiAgICBtb3YuZjMyICVjMSwgMGYwMDAwMDAwMDsKICAgIG1vdi5mMzIgJWMyLCAwZjAwMDAwMDAwOwogICAgbW92LmYzMiAlYzMsIDBmMDAwMDAwMDA7CiAgICBtb3YuZjMyICVjNCwgMGYwMDAwMDAwMDsKICAgIG1vdi5mMzIgJWM1LCAwZjAwMDAwMDAwOwogICAgbW92LmYzMiAlYzYsIDBmMDAwMDAwMDA7CiAgICBtb3YuZjMyICVjNywgMGYwMDAwMDAwMDsKICAgIG1vdi51MzIgJXIxMSwgMDsKClc2NF9NTUFfSzoKICAgIHNldHAuZ2UudTMyICVwMSwgJXIxMSwgJXIyOwogICAgQCVwMSBicmEgVzY0X01NQV9TVE9SRTsKICAgIHNobC5iMzIgJXIxMiwgJXI3LCAxOwogICAgYWRkLnUzMiAlcjEyLCAlcjEyLCAlcjExOwoKICAgIC8vIEEgZnJhZ21lbnQ6IHByb2JhYmlsaXR5IHJvd3MgZ3JvdXBJRCBhbmQgZ3JvdXBJRCs4LCB0d28gSyB2YWx1ZXMuCiAgICBhZGQudTMyICVyMTMsICVyOCwgJXI2OwogICAgbXVsLmxvLnUzMiAlcjE0LCAlcjksICVyMTsKICAgIGFkZC51MzIgJXIxNCwgJXIxNCwgJXIxMzsKICAgIG11bC5sby51MzIgJXIxNCwgJXIxNCwgJXIyOwogICAgYWRkLnUzMiAlcjE0LCAlcjE0LCAlcjEyOwogICAgc2hsLmIzMiAlcjE0LCAlcjE0LCAyOwogICAgY3Z0LnU2NC51MzIgJXJkNywgJXIxNDsKICAgIGFkZC51NjQgJXJkNywgJXJkNCwgJXJkNzsKICAgIG1vdi5mMzIgJWYxLCAwZjAwMDAwMDAwOwogICAgbW92LmYzMiAlZjIsIDBmMDAwMDAwMDA7CiAgICBzZXRwLmx0LnUzMiAlcDIsICVyMTIsICVyMjsKICAgIHNldHAubHQudTMyICVwMywgJXIxMywgJXIxOwogICAgYW5kLnByZWQgJXA0LCAlcDIsICVwMzsKICAgIEAlcDQgbGQuZ2xvYmFsLmYzMiAlZjEsIFslcmQ3XTsKICAgIGFkZC51MzIgJXIxNSwgJXIxMywgODsKICAgIHNldHAubHQudTMyICVwMywgJXIxNSwgJXIxOwogICAgYW5kLnByZWQgJXA0LCAlcDIsICVwMzsKICAgIEAhJXA0IGJyYSBXNjRfQV9ST1c4X1pFUk87CiAgICBtdWwubG8udTMyICVyMTYsICVyOSwgJXIxOwogICAgYWRkLnUzMiAlcjE2LCAlcjE2LCAlcjE1OwogICAgbXVsLmxvLnUzMiAlcjE2LCAlcjE2LCAlcjI7CiAgICBhZGQudTMyICVyMTYsICVyMTYsICVyMTI7CiAgICBzaGwuYjMyICVyMTYsICVyMTYsIDI7CiAgICBjdnQudTY0LnUzMiAlcmQ4LCAlcjE2OwogICAgYWRkLnU2NCAlcmQ4LCAlcmQ0LCAlcmQ4OwogICAgbGQuZ2xvYmFsLmYzMiAlZjIsIFslcmQ4XTsKVzY0X0FfUk9XOF9aRVJPOgogICAgY3Z0LnJuLmYxNi5mMzIgJWgwLCAlZjE7CiAgICBjdnQucm4uZjE2LmYzMiAlaDEsICVmMjsKICAgIGN2dC5mMzIuZjE2ICVmMywgJWgwOwogICAgY3Z0LmYzMi5mMTYgJWY0LCAlaDE7CiAgICBzdWIucm4uZjMyICVmNSwgJWYxLCAlZjM7CiAgICBzdWIucm4uZjMyICVmNiwgJWYyLCAlZjQ7CiAgICBjdnQucm4uZjE2LmYzMiAlaDIsICVmNTsKICAgIGN2dC5ybi5mMTYuZjMyICVoMywgJWY2OwogICAgLy8gRWFjaCByZWdpc3RlciBwYWNrcyB0d28gYWRqYWNlbnQgSyB2YWx1ZXMgZm9yIG9uZSByb3cgZnJhZ21lbnQuIFRoZQogICAgLy8gc2Vjb25kIHZhbHVlIGlzIGxvYWRlZCBleHBsaWNpdGx5IHRvIHByZXNlcnZlIHRoZSBNTUEgbGFuZSBjb250cmFjdC4KICAgIGFkZC51MzIgJXIxNywgJXIxMiwgMTsKICAgIG1vdi5mMzIgJWY3LCAwZjAwMDAwMDAwOwogICAgbW92LmYzMiAlZjgsIDBmMDAwMDAwMDA7CiAgICBzZXRwLmx0LnUzMiAlcDIsICVyMTcsICVyMjsKICAgIHNldHAubHQudTMyICVwMywgJXIxMywgJXIxOwogICAgYW5kLnByZWQgJXA0LCAlcDIsICVwMzsKICAgIEAlcDQgbGQuZ2xvYmFsLmYzMiAlZjcsIFslcmQ3KzRdOwogICAgc2V0cC5sdC51MzIgJXAzLCAlcjE1LCAlcjE7CiAgICBhbmQucHJlZCAlcDQsICVwMiwgJXAzOwogICAgQCVwNCBsZC5nbG9iYWwuZjMyICVmOCwgWyVyZDgrNF07CiAgICBjdnQucm4uZjE2LmYzMiAlaDQsICVmNzsKICAgIGN2dC5ybi5mMTYuZjMyICVoNSwgJWY4OwogICAgY3Z0LmYzMi5mMTYgJWY5LCAlaDQ7CiAgICBjdnQuZjMyLmYxNiAlZjEwLCAlaDU7CiAgICBzdWIucm4uZjMyICVmMTEsICVmNywgJWY5OwogICAgc3ViLnJuLmYzMiAlZjEyLCAlZjgsICVmMTA7CiAgICBjdnQucm4uZjE2LmYzMiAlaDYsICVmMTE7CiAgICBjdnQucm4uZjE2LmYzMiAlaDcsICVmMTI7CiAgICBtb3YuYjMyICVhX2hpMCwgeyVoMCwgJWg0fTsKICAgIG1vdi5iMzIgJWFfaGkxLCB7JWgxLCAlaDV9OwogICAgbW92LmIzMiAlYV9sbzAsIHslaDIsICVoNn07CiAgICBtb3YuYjMyICVhX2xvMSwgeyVoMywgJWg3fTsKCiAgICAvLyBCIGZyYWdtZW50czogdHdvIGFkamFjZW50IE44IHRpbGVzIG93bmVkIGJ5IHRoaXMgd2FycC4gUmV0YWluZWQgViBpcwogICAgLy8gW2hlYWQsIEssIDY0XS4gQWNyb3NzIHRoZSB3YXJwLCBncm91cElEIHNlbGVjdHMgTiBhbmQgdGhyZWFkSW5Hcm91cAogICAgLy8gc2VsZWN0cyB0aGUgSyBwYWlyLCBjb3ZlcmluZyB0aGUgY29tcGxldGUgOHg4IG9wZXJhbmQgd2l0aG91dCBhCiAgICAvLyBwZXJtYW5lbnQgdHJhbnNwb3NlLgogICAgYWRkLnUzMiAlcjE4LCAlcjEwLCAlcjY7CiAgICBtdWwubG8udTMyICVyMTksICVyOSwgJXIyOwogICAgYWRkLnUzMiAlcjE5LCAlcjE5LCAlcjEyOwogICAgc2hsLmIzMiAlcjE5LCAlcjE5LCA2OwogICAgYWRkLnUzMiAlcjE5LCAlcjE5LCAlcjE4OwogICAgc2hsLmIzMiAlcjE5LCAlcjE5LCAyOwogICAgY3Z0LnU2NC51MzIgJXJkOSwgJXIxOTsKICAgIGFkZC51NjQgJXJkOSwgJXJkNSwgJXJkOTsKICAgIG1vdi5mMzIgJWYxMywgMGYwMDAwMDAwMDsKICAgIG1vdi5mMzIgJWYxNCwgMGYwMDAwMDAwMDsKICAgIHNldHAubHQudTMyICVwMiwgJXIxMiwgJXIyOwogICAgQCVwMiBsZC5nbG9iYWwuZjMyICVmMTMsIFslcmQ5XTsKICAgIHNldHAubHQudTMyICVwMiwgJXIxNywgJXIyOwogICAgQCVwMiBsZC5nbG9iYWwuZjMyICVmMTQsIFslcmQ5KzI1Nl07CiAgICBjdnQucm4uZjE2LmYzMiAlaDgsICVmMTM7CiAgICBjdnQucm4uZjE2LmYzMiAlaDksICVmMTQ7CiAgICBjdnQuZjMyLmYxNiAlZjE1LCAlaDg7CiAgICBjdnQuZjMyLmYxNiAlZjE2LCAlaDk7CiAgICBzdWIucm4uZjMyICVmMTcsICVmMTMsICVmMTU7CiAgICBzdWIucm4uZjMyICVmMTgsICVmMTQsICVmMTY7CiAgICBjdnQucm4uZjE2LmYzMiAlaDEwLCAlZjE3OwogICAgY3Z0LnJuLmYxNi5mMzIgJWgxMSwgJWYxODsKICAgIG1vdi5iMzIgJWIwX2hpLCB7JWg4LCAlaDl9OwogICAgbW92LmIzMiAlYjBfbG8sIHslaDEwLCAlaDExfTsKCiAgICBhZGQudTMyICVyMjAsICVyMTgsIDg7CiAgICBtdWwubG8udTMyICVyMjEsICVyOSwgJXIyOwogICAgYWRkLnUzMiAlcjIxLCAlcjIxLCAlcjEyOwogICAgc2hsLmIzMiAlcjIxLCAlcjIxLCA2OwogICAgYWRkLnUzMiAlcjIxLCAlcjIxLCAlcjIwOwogICAgc2hsLmIzMiAlcjIxLCAlcjIxLCAyOwogICAgY3Z0LnU2NC51MzIgJXJkMTAsICVyMjE7CiAgICBhZGQudTY0ICVyZDEwLCAlcmQ1LCAlcmQxMDsKICAgIG1vdi5mMzIgJWYxMywgMGYwMDAwMDAwMDsKICAgIG1vdi5mMzIgJWYxNCwgMGYwMDAwMDAwMDsKICAgIHNldHAubHQudTMyICVwMiwgJXIxMiwgJXIyOwogICAgQCVwMiBsZC5nbG9iYWwuZjMyICVmMTMsIFslcmQxMF07CiAgICBzZXRwLmx0LnUzMiAlcDIsICVyMTcsICVyMjsKICAgIEAlcDIgbGQuZ2xvYmFsLmYzMiAlZjE0LCBbJXJkMTArMjU2XTsKICAgIGN2dC5ybi5mMTYuZjMyICVoMTIsICVmMTM7CiAgICBjdnQucm4uZjE2LmYzMiAlaDEzLCAlZjE0OwogICAgY3Z0LmYzMi5mMTYgJWYxNSwgJWgxMjsKICAgIGN2dC5mMzIuZjE2ICVmMTYsICVoMTM7CiAgICBzdWIucm4uZjMyICVmMTcsICVmMTMsICVmMTU7CiAgICBzdWIucm4uZjMyICVmMTgsICVmMTQsICVmMTY7CiAgICBjdnQucm4uZjE2LmYzMiAlaDE0LCAlZjE3OwogICAgY3Z0LnJuLmYxNi5mMzIgJWgxNSwgJWYxODsKICAgIG1vdi5iMzIgJWIxX2hpLCB7JWgxMiwgJWgxM307CiAgICBtb3YuYjMyICViMV9sbywgeyVoMTQsICVoMTV9OwoKICAgIG1tYS5zeW5jLmFsaWduZWQubTE2bjhrOC5yb3cuY29sLmYzMi5mMTYuZjE2LmYzMgogICAgICAgIHslYzAsJWMxLCVjMiwlYzN9LCB7JWFfaGkwLCVhX2hpMX0sIHslYjBfaGl9LCB7JWMwLCVjMSwlYzIsJWMzfTsKICAgIG1tYS5zeW5jLmFsaWduZWQubTE2bjhrOC5yb3cuY29sLmYzMi5mMTYuZjE2LmYzMgogICAgICAgIHslYzAsJWMxLCVjMiwlYzN9LCB7JWFfaGkwLCVhX2hpMX0sIHslYjBfbG99LCB7JWMwLCVjMSwlYzIsJWMzfTsKICAgIG1tYS5zeW5jLmFsaWduZWQubTE2bjhrOC5yb3cuY29sLmYzMi5mMTYuZjE2LmYzMgogICAgICAgIHslYzAsJWMxLCVjMiwlYzN9LCB7JWFfbG8wLCVhX2xvMX0sIHslYjBfaGl9LCB7JWMwLCVjMSwlYzIsJWMzfTsKICAgIG1tYS5zeW5jLmFsaWduZWQubTE2bjhrOC5yb3cuY29sLmYzMi5mMTYuZjE2LmYzMgogICAgICAgIHslYzAsJWMxLCVjMiwlYzN9LCB7JWFfbG8wLCVhX2xvMX0sIHslYjBfbG99LCB7JWMwLCVjMSwlYzIsJWMzfTsKICAgIG1tYS5zeW5jLmFsaWduZWQubTE2bjhrOC5yb3cuY29sLmYzMi5mMTYuZjE2LmYzMgogICAgICAgIHslYzQsJWM1LCVjNiwlYzd9LCB7JWFfaGkwLCVhX2hpMX0sIHslYjFfaGl9LCB7JWM0LCVjNSwlYzYsJWM3fTsKICAgIG1tYS5zeW5jLmFsaWduZWQubTE2bjhrOC5yb3cuY29sLmYzMi5mMTYuZjE2LmYzMgogICAgICAgIHslYzQsJWM1LCVjNiwlYzd9LCB7JWFfaGkwLCVhX2hpMX0sIHslYjFfbG99LCB7JWM0LCVjNSwlYzYsJWM3fTsKICAgIG1tYS5zeW5jLmFsaWduZWQubTE2bjhrOC5yb3cuY29sLmYzMi5mMTYuZjE2LmYzMgogICAgICAgIHslYzQsJWM1LCVjNiwlYzd9LCB7JWFfbG8wLCVhX2xvMX0sIHslYjFfaGl9LCB7JWM0LCVjNSwlYzYsJWM3fTsKICAgIG1tYS5zeW5jLmFsaWduZWQubTE2bjhrOC5yb3cuY29sLmYzMi5mMTYuZjE2LmYzMgogICAgICAgIHslYzQsJWM1LCVjNiwlYzd9LCB7JWFfbG8wLCVhX2xvMX0sIHslYjFfbG99LCB7JWM0LCVjNSwlYzYsJWM3fTsKICAgIGFkZC51MzIgJXIxMSwgJXIxMSwgODsKICAgIGJyYSBXNjRfTU1BX0s7CgpXNjRfTU1BX1NUT1JFOgogICAgYWRkLnUzMiAlcjIyLCAlcjgsICVyNjsKICAgIHNldHAuZ2UudTMyICVwNSwgJXIyMiwgJXIxOwogICAgQCVwNSBicmEgVzY0X01NQV9ET05FOwogICAgc2hsLmIzMiAlcjIzLCAlcjcsIDE7CiAgICBhZGQudTMyICVyMjQsICVyMTAsICVyMjM7CiAgICBtdWwubG8udTMyICVyMjUsICVyOSwgJXIxOwogICAgYWRkLnUzMiAlcjI1LCAlcjI1LCAlcjIyOwogICAgc2hsLmIzMiAlcjI1LCAlcjI1LCA2OwogICAgYWRkLnUzMiAlcjI1LCAlcjI1LCAlcjI0OwogICAgc2hsLmIzMiAlcjI1LCAlcjI1LCAyOwogICAgY3Z0LnU2NC51MzIgJXJkMTEsICVyMjU7CiAgICBhZGQudTY0ICVyZDExLCAlcmQ2LCAlcmQxMTsKICAgIHN0Lmdsb2JhbC5mMzIgWyVyZDExXSwgJWMwOwogICAgc3QuZ2xvYmFsLmYzMiBbJXJkMTErNF0sICVjMTsKICAgIHN0Lmdsb2JhbC5mMzIgWyVyZDExKzMyXSwgJWM0OwogICAgc3QuZ2xvYmFsLmYzMiBbJXJkMTErMzZdLCAlYzU7CiAgICBhZGQudTMyICVyMjIsICVyMjIsIDg7CiAgICBzZXRwLmdlLnUzMiAlcDUsICVyMjIsICVyMTsKICAgIEAlcDUgYnJhIFc2NF9NTUFfRE9ORTsKICAgIGFkZC51NjQgJXJkMTEsICVyZDExLCAyMDQ4OwogICAgc3QuZ2xvYmFsLmYzMiBbJXJkMTFdLCAlYzI7CiAgICBzdC5nbG9iYWwuZjMyIFslcmQxMSs0XSwgJWMzOwogICAgc3QuZ2xvYmFsLmYzMiBbJXJkMTErMzJdLCAlYzY7CiAgICBzdC5nbG9iYWwuZjMyIFslcmQxMSszNl0sICVjNzsKVzY0X01NQV9ET05FOgogICAgcmV0Owp9Cg=="},{"path":"glcuda/examples/wave64_mma_av.rs","sha256":"a7ec0964c8728d8d092894be50f702199ae5ece7e38887b7e3d8f857a8056dcf","base64":"Ly8hIFdhdmUgNjQgY29tcGVuc2F0ZWQtTU1BIEFWIGZlYXNpYmlsaXR5IHNjcmVlbi4KLy8hCi8vISBUaGlzIGxvYWRzIGFuIGlzb2xhdGVkIFBUWCBtb2R1bGUuIEl0IGRvZXMgbm90IGFsdGVyIEtlcm5lbFNldCBvciB0aGUKLy8hIHByb2R1Y3Rpb24gYXR0ZW50aW9uIGRpc3BhdGNoZXIuCgp1c2Ugc3RkOjpmZmk6OmNfdm9pZDsKdXNlIHN0ZDo6dGltZTo6SW5zdGFudDsKCnVzZSBnbGN1ZGE6OmJ1ZmZlcjo6QmFja2VuZEJ1ZmZlcjsKdXNlIGdsY3VkYTo6ZHJpdmVyOjp7Y3VkYV9hdmFpbGFibGUsIEN1ZGEsIEtlcm5lbH07CnVzZSBnbGN1ZGE6OmZmaTo6Q1VkZXZpY2VwdHI7Cgpjb25zdCBIRUFEUzogdXNpemUgPSAxNDsKY29uc3QgV0lEVEg6IHVzaXplID0gNjQ7CmNvbnN0IFdBUk1VUDogdXNpemUgPSAxMDsKY29uc3QgSVRFUlM6IHVzaXplID0gMTAwOwpjb25zdCBSRVBFQVRTOiB1c2l6ZSA9IDU7CmNvbnN0IE1BWF9BQlNfR0FURTogZjMyID0gMS4wZS01Owpjb25zdCBNSU5fUFJPRFVDVElPTl9TUEVFRFVQOiBmNjQgPSAxLjUwOwpjb25zdCBQVFg6ICZzdHIgPSBpbmNsdWRlX3N0ciEoIi4uL3NyYy9rZXJuZWxzL2dsY3VkYV9zbTc1X3dhdmU2NC5wdHgiKTsKCmZuIHZhbHVlcyhuOiB1c2l6ZSwgc2VlZDogdTY0KSAtPiBWZWM8ZjMyPiB7CiAgICBsZXQgbXV0IHN0YXRlID0gc2VlZCB8IDE7CiAgICAoMC4ubikKICAgICAgICAubWFwKHxffCB7CiAgICAgICAgICAgIHN0YXRlIF49IHN0YXRlID4+IDEyOwogICAgICAgICAgICBzdGF0ZSBePSBzdGF0ZSA8PCAyNTsKICAgICAgICAgICAgc3RhdGUgXj0gc3RhdGUgPj4gMjc7CiAgICAgICAgICAgIGxldCB1bml0ID0KICAgICAgICAgICAgICAgIChzdGF0ZS53cmFwcGluZ19tdWwoMHgyNTQ1X0Y0OTFfNEY2Q19ERDFEKSA+PiA0MCkgYXMgZjMyIC8gKDF1NjQgPDwgMjQpIGFzIGYzMjsKICAgICAgICAgICAgdW5pdCAtIDAuNQogICAgICAgIH0pCiAgICAgICAgLmNvbGxlY3QoKQp9CgpmbiBjYXVzYWxfcHJvYmFiaWxpdGllcyhudG9rOiB1c2l6ZSwgY2FwYWNpdHk6IHVzaXplKSAtPiBWZWM8ZjMyPiB7CiAgICBsZXQgcmF3ID0gdmFsdWVzKEhFQURTICogbnRvayAqIGNhcGFjaXR5LCA2NCArIGNhcGFjaXR5IGFzIHU2NCk7CiAgICBsZXQgbXV0IG91dCA9IHZlYyFbMC4wOyByYXcubGVuKCldOwogICAgZm9yIGhlYWQgaW4gMC4uSEVBRFMgewogICAgICAgIGZvciByb3cgaW4gMC4ubnRvayB7CiAgICAgICAgICAgIGxldCB2aXNpYmxlID0gKHJvdyArIDEpLm1pbihjYXBhY2l0eSk7CiAgICAgICAgICAgIGxldCBiYXNlID0gKGhlYWQgKiBudG9rICsgcm93KSAqIGNhcGFjaXR5OwogICAgICAgICAgICBsZXQgbWF4ID0gcmF3W2Jhc2UuLmJhc2UgKyB2aXNpYmxlXQogICAgICAgICAgICAgICAgLml0ZXIoKQogICAgICAgICAgICAgICAgLmNvcGllZCgpCiAgICAgICAgICAgICAgICAuZm9sZChmMzI6Ok5FR19JTkZJTklUWSwgZjMyOjptYXgpOwogICAgICAgICAgICBsZXQgbXV0IHN1bSA9IDAuMGYzMjsKICAgICAgICAgICAgZm9yIGtleSBpbiAwLi52aXNpYmxlIHsKICAgICAgICAgICAgICAgIGxldCB3ZWlnaHQgPSAocmF3W2Jhc2UgKyBrZXldIC0gbWF4KS5leHAoKTsKICAgICAgICAgICAgICAgIG91dFtiYXNlICsga2V5XSA9IHdlaWdodDsKICAgICAgICAgICAgICAgIHN1bSArPSB3ZWlnaHQ7CiAgICAgICAgICAgIH0KICAgICAgICAgICAgZm9yIGtleSBpbiAwLi52aXNpYmxlIHsKICAgICAgICAgICAgICAgIG91dFtiYXNlICsga2V5XSAvPSBzdW07CiAgICAgICAgICAgIH0KICAgICAgICB9CiAgICB9CiAgICBvdXQKfQoKZm4gbGF1bmNoX2F2KAogICAgY3VkYTogJkN1ZGEsCiAgICBrZXJuZWw6IEtlcm5lbCwKICAgIHByb2I6IENVZGV2aWNlcHRyLAogICAgdjogQ1VkZXZpY2VwdHIsCiAgICBvdXQ6IENVZGV2aWNlcHRyLAogICAgbnRvazogdTMyLAogICAgY2FwYWNpdHk6IHUzMiwKKSAtPiBSZXN1bHQ8KCksIGdsY29yZTo6R2xFcnJvcj4gewogICAgbGV0IChtdXQgcHJvYiwgbXV0IHYsIG11dCBvdXQsIG11dCBudG9rLCBtdXQgY2FwYWNpdHkpID0gKHByb2IsIHYsIG91dCwgbnRvaywgY2FwYWNpdHkpOwogICAgbGV0IG11dCBwYXJhbXMgPSBbCiAgICAgICAgJm11dCBwcm9iIGFzICptdXQgXyBhcyAqbXV0IGNfdm9pZCwKICAgICAgICAmbXV0IHYgYXMgKm11dCBfIGFzICptdXQgY192b2lkLAogICAgICAgICZtdXQgb3V0IGFzICptdXQgXyBhcyAqbXV0IGNfdm9pZCwKICAgICAgICAmbXV0IG50b2sgYXMgKm11dCBfIGFzICptdXQgY192b2lkLAogICAgICAgICZtdXQgY2FwYWNpdHkgYXMgKm11dCBfIGFzICptdXQgY192b2lkLAogICAgXTsKICAgIGN1ZGEubGF1bmNoKAogICAgICAgIGtlcm5lbCwKICAgICAgICAobnRvay5kaXZfY2VpbCgxNiksIEhFQURTIGFzIHUzMiwgMSksCiAgICAgICAgKDEyOCwgMSwgMSksCiAgICAgICAgMCwKICAgICAgICAmbXV0IHBhcmFtcywKICAgICkKfQoKZm4gdGltZWQ8Rj4oY3VkYTogJkN1ZGEsIG11dCBsYXVuY2g6IEYpIC0+IFJlc3VsdDxmNjQsIGdsY29yZTo6R2xFcnJvcj4Kd2hlcmUKICAgIEY6IEZuTXV0KCkgLT4gUmVzdWx0PCgpLCBnbGNvcmU6OkdsRXJyb3I+LAp7CiAgICBmb3IgXyBpbiAwLi5XQVJNVVAgewogICAgICAgIGxhdW5jaCgpPzsKICAgIH0KICAgIGN1ZGEuc3luY2hyb25pemUoKT87CiAgICBsZXQgc3RhcnQgPSBJbnN0YW50Ojpub3coKTsKICAgIGZvciBfIGluIDAuLklURVJTIHsKICAgICAgICBsYXVuY2goKT87CiAgICB9CiAgICBjdWRhLnN5bmNocm9uaXplKCk/OwogICAgT2soc3RhcnQuZWxhcHNlZCgpLmFzX3NlY3NfZjY0KCkgKiAxLjBlNiAvIElURVJTIGFzIGY2NCkKfQoKZm4gbWVkaWFuKHNhbXBsZXM6ICZtdXQgW2Y2NF0pIC0+IGY2NCB7CiAgICBzYW1wbGVzLnNvcnRfYnkoZjY0Ojp0b3RhbF9jbXApOwogICAgKHNhbXBsZXNbc2FtcGxlcy5sZW4oKSAvIDIgLSAxXSArIHNhbXBsZXNbc2FtcGxlcy5sZW4oKSAvIDJdKSAqIDAuNQp9CgpzdHJ1Y3QgUmVjb3JkIHsKICAgIGNhcGFjaXR5OiB1c2l6ZSwKICAgIHNjYWxhcl91czogZjY0LAogICAgbW1hX3VzOiBmNjQsCiAgICBtYXhfYWJzOiBmMzIsCiAgICBybXM6IGY2NCwKfQoKaW1wbCBSZWNvcmQgewogICAgZm4ganNvbigmc2VsZikgLT4gU3RyaW5nIHsKICAgICAgICBmb3JtYXQhKAogICAgICAgICAgICAie3tcImNhcGFjaXR5XCI6e30sXCJzY2FsYXJfdXNcIjp7Oi4zfSxcIm1tYV91c1wiOns6LjN9LFwKICAgICAgICAgICAgIFwic3BlZWR1cFwiOns6LjR9LFwibWF4X2Fic1wiOns6LjllfSxcInJtc1wiOns6LjllfX19IiwKICAgICAgICAgICAgc2VsZi5jYXBhY2l0eSwKICAgICAgICAgICAgc2VsZi5zY2FsYXJfdXMsCiAgICAgICAgICAgIHNlbGYubW1hX3VzLAogICAgICAgICAgICBzZWxmLnNjYWxhcl91cyAvIHNlbGYubW1hX3VzLAogICAgICAgICAgICBzZWxmLm1heF9hYnMsCiAgICAgICAgICAgIHNlbGYucm1zLAogICAgICAgICkKICAgIH0KfQoKZm4gc2NyZWVuKAogICAgY3VkYTogJkN1ZGEsCiAgICBzY2FsYXI6IEtlcm5lbCwKICAgIG1tYTogS2VybmVsLAogICAgY2FwYWNpdHk6IHVzaXplLAopIC0+IFJlc3VsdDxSZWNvcmQsIEJveDxkeW4gc3RkOjplcnJvcjo6RXJyb3I+PiB7CiAgICBsZXQgbnRvayA9IGNhcGFjaXR5OwogICAgbGV0IHByb2IgPSBjYXVzYWxfcHJvYmFiaWxpdGllcyhudG9rLCBjYXBhY2l0eSk7CiAgICBsZXQgdiA9IHZhbHVlcyhIRUFEUyAqIGNhcGFjaXR5ICogV0lEVEgsIDY0MDAgKyBjYXBhY2l0eSBhcyB1NjQpOwogICAgbGV0IG91dHB1dF9sZW4gPSBIRUFEUyAqIG50b2sgKiBXSURUSDsKICAgIGxldCBieXRlcyA9ICgocHJvYi5sZW4oKSArIHYubGVuKCkgKyAyICogb3V0cHV0X2xlbikgKiA0ICsgMV8wNDhfNTc2KSBhcyB1NjQ7CiAgICBsZXQgbXV0IGJ1ZmZlciA9IEJhY2tlbmRCdWZmZXI6Om5ldyhjdWRhLCBieXRlcyk/OwogICAgbGV0IGRwcm9iID0gYnVmZmVyLmFsbG9jX2YzMihwcm9iLmxlbigpKT8uZHB0cjsKICAgIGxldCBkdiA9IGJ1ZmZlci5hbGxvY19mMzIodi5sZW4oKSk/LmRwdHI7CiAgICBsZXQgZHNjYWxhciA9IGJ1ZmZlci5hbGxvY19mMzIob3V0cHV0X2xlbik/LmRwdHI7CiAgICBsZXQgZG1tYSA9IGJ1ZmZlci5hbGxvY19mMzIob3V0cHV0X2xlbik/LmRwdHI7CiAgICBjdWRhLmh0b2RfZjMyKGRwcm9iLCAmcHJvYik/OwogICAgY3VkYS5odG9kX2YzMihkdiwgJnYpPzsKCiAgICBsZXQgc2NhbGFyX2xhdW5jaCA9IHx8IHsKICAgICAgICBsYXVuY2hfYXYoCiAgICAgICAgICAgIGN1ZGEsCiAgICAgICAgICAgIHNjYWxhciwKICAgICAgICAgICAgZHByb2IsCiAgICAgICAgICAgIGR2LAogICAgICAgICAgICBkc2NhbGFyLAogICAgICAgICAgICBudG9rIGFzIHUzMiwKICAgICAgICAgICAgY2FwYWNpdHkgYXMgdTMyLAogICAgICAgICkKICAgIH07CiAgICBsZXQgbW1hX2xhdW5jaCA9IHx8IGxhdW5jaF9hdihjdWRhLCBtbWEsIGRwcm9iLCBkdiwgZG1tYSwgbnRvayBhcyB1MzIsIGNhcGFjaXR5IGFzIHUzMik7CiAgICBzY2FsYXJfbGF1bmNoKCk/OwogICAgbW1hX2xhdW5jaCgpPzsKICAgIGN1ZGEuc3luY2hyb25pemUoKT87CiAgICBsZXQgbXV0IHNjYWxhcl9ob3N0ID0gdmVjIVswLjA7IG91dHB1dF9sZW5dOwogICAgbGV0IG11dCBtbWFfaG9zdCA9IHZlYyFbMC4wOyBvdXRwdXRfbGVuXTsKICAgIGN1ZGEuZHRvaF9mMzIoJm11dCBzY2FsYXJfaG9zdCwgZHNjYWxhcik/OwogICAgY3VkYS5kdG9oX2YzMigmbXV0IG1tYV9ob3N0LCBkbW1hKT87CiAgICBpZiAhc2NhbGFyX2hvc3QKICAgICAgICAuaXRlcigpCiAgICAgICAgLmNoYWluKCZtbWFfaG9zdCkKICAgICAgICAuYWxsKHx2YWx1ZXwgdmFsdWUuaXNfZmluaXRlKCkpCiAgICB7CiAgICAgICAgcmV0dXJuIEVycihmb3JtYXQhKCJub24tZmluaXRlIFdhdmUgNjQgb3V0cHV0IGF0IGNhcGFjaXR5IHtjYXBhY2l0eX0iKS5pbnRvKCkpOwogICAgfQogICAgbGV0IG11dCBtYXhfYWJzID0gMC4wZjMyOwogICAgbGV0IG11dCBzcXVhcmVkID0gMC4wZjY0OwogICAgZm9yICgmbGVmdCwgJnJpZ2h0KSBpbiBzY2FsYXJfaG9zdC5pdGVyKCkuemlwKCZtbWFfaG9zdCkgewogICAgICAgIGxldCBkZWx0YSA9IChsZWZ0IC0gcmlnaHQpLmFicygpOwogICAgICAgIG1heF9hYnMgPSBtYXhfYWJzLm1heChkZWx0YSk7CiAgICAgICAgc3F1YXJlZCArPSBmNjQ6OmZyb20oZGVsdGEpLnBvd2koMik7CiAgICB9CiAgICBsZXQgcm1zID0gKHNxdWFyZWQgLyBvdXRwdXRfbGVuIGFzIGY2NCkuc3FydCgpOwoKICAgIGxldCAobXV0IHNjYWxhcl9zYW1wbGVzLCBtdXQgbW1hX3NhbXBsZXMpID0gKAogICAgICAgIFZlYzo6d2l0aF9jYXBhY2l0eSgyICogUkVQRUFUUyksCiAgICAgICAgVmVjOjp3aXRoX2NhcGFjaXR5KDIgKiBSRVBFQVRTKSwKICAgICk7CiAgICBmb3IgXyBpbiAwLi5SRVBFQVRTIHsKICAgICAgICBzY2FsYXJfc2FtcGxlcy5wdXNoKHRpbWVkKGN1ZGEsIHNjYWxhcl9sYXVuY2gpPyk7CiAgICAgICAgbW1hX3NhbXBsZXMucHVzaCh0aW1lZChjdWRhLCBtbWFfbGF1bmNoKT8pOwogICAgICAgIG1tYV9zYW1wbGVzLnB1c2godGltZWQoY3VkYSwgbW1hX2xhdW5jaCk/KTsKICAgICAgICBzY2FsYXJfc2FtcGxlcy5wdXNoKHRpbWVkKGN1ZGEsIHNjYWxhcl9sYXVuY2gpPyk7CiAgICB9CiAgICBsZXQgc2NhbGFyX3VzID0gbWVkaWFuKCZtdXQgc2NhbGFyX3NhbXBsZXMpOwogICAgbGV0IG1tYV91cyA9IG1lZGlhbigmbXV0IG1tYV9zYW1wbGVzKTsKICAgIGJ1ZmZlci5mcmVlKGN1ZGEpPzsKICAgIE9rKFJlY29yZCB7CiAgICAgICAgY2FwYWNpdHksCiAgICAgICAgc2NhbGFyX3VzLAogICAgICAgIG1tYV91cywKICAgICAgICBtYXhfYWJzLAogICAgICAgIHJtcywKICAgIH0pCn0KCmZuIG1haW4oKSAtPiBSZXN1bHQ8KCksIEJveDxkeW4gc3RkOjplcnJvcjo6RXJyb3I+PiB7CiAgICBpZiAhY3VkYV9hdmFpbGFibGUoKSB7CiAgICAgICAgcHJpbnRsbiEoIlt3YXZlNjQtYXZdIG5vIENVREEgZGV2aWNlOyBub3RoaW5nIG1lYXN1cmVkIik7CiAgICAgICAgcmV0dXJuIE9rKCgpKTsKICAgIH0KICAgIGxldCBjdWRhID0gQ3VkYTo6cHJvYmUoKT87CiAgICBpZiAoY3VkYS5pbmZvLnNtX21ham9yLCBjdWRhLmluZm8uc21fbWlub3IpICE9ICg3LCA1KSB7CiAgICAgICAgcmV0dXJuIEVycihmb3JtYXQhKAogICAgICAgICAgICAiV2F2ZSA2NCByZXF1aXJlcyBzbV83NSwgZ290IHNtX3t9e30iLAogICAgICAgICAgICBjdWRhLmluZm8uc21fbWFqb3IsIGN1ZGEuaW5mby5zbV9taW5vcgogICAgICAgICkKICAgICAgICAuaW50bygpKTsKICAgIH0KICAgIGxldCBtb2R1bGUgPSBjdWRhLmxvYWRfbW9kdWxlKFBUWCk/OwogICAgbGV0IHNjYWxhciA9IG1vZHVsZS5nZXRfZnVuY3Rpb24oImdsX3dhdmU2NF9hdl9zY2FsYXJfZjMyIik/OwogICAgbGV0IG1tYSA9IG1vZHVsZS5nZXRfZnVuY3Rpb24oImdsX3dhdmU3NV9hdl9tbWE0X3Jvd19mMzIiKT87CiAgICBsZXQgcmVjb3JkcyA9IFsKICAgICAgICBzY3JlZW4oJmN1ZGEsIHNjYWxhciwgbW1hLCAxKT8sCiAgICAgICAgc2NyZWVuKCZjdWRhLCBzY2FsYXIsIG1tYSwgMTcpPywKICAgICAgICBzY3JlZW4oJmN1ZGEsIHNjYWxhciwgbW1hLCAyNDEpPywKICAgICAgICBzY3JlZW4oJmN1ZGEsIHNjYWxhciwgbW1hLCAyNDQpPywKICAgIF07CiAgICBwcmludGxuISgKICAgICAgICAiW3dhdmU2NC1yZXNvdXJjZV0ge3tcInRocmVhZHNcIjoxMjgsXCJkeW5hbWljX3NoYXJlZF9ieXRlc1wiOjAsXAogICAgICAgICBcIm9jY3VwYW5jeV9zb3VyY2VcIjpcInB0eGFzLW9ubHlcIn19IiwKICAgICk7CiAgICBwcmludGxuISgKICAgICAgICAiW3dhdmU2NC1hdl0ge3tcIndhcm11cFwiOnt9LFwiaXRlcnNcIjp7fSxcInJlcGVhdHNcIjp7fSxcInJlY29yZHNcIjpbe31dfX0iLAogICAgICAgIFdBUk1VUCwKICAgICAgICBJVEVSUywKICAgICAgICBSRVBFQVRTLAogICAgICAgIHJlY29yZHMKICAgICAgICAgICAgLml0ZXIoKQogICAgICAgICAgICAubWFwKFJlY29yZDo6anNvbikKICAgICAgICAgICAgLmNvbGxlY3Q6OjxWZWM8Xz4+KCkKICAgICAgICAgICAgLmpvaW4oIiwiKSwKICAgICk7CiAgICBpZiBsZXQgU29tZShyZWNvcmQpID0gcmVjb3Jkcy5pdGVyKCkuZmluZCh8cmVjb3JkfCByZWNvcmQubWF4X2FicyA+IE1BWF9BQlNfR0FURSkgewogICAgICAgIHJldHVybiBFcnIoZm9ybWF0ISgKICAgICAgICAgICAgIldhdmUgNjQgbnVtZXJpYyBnYXRlIGZhaWxlZCBhdCBjYXBhY2l0eSB7fTogezouOWV9ID4gezouMWV9IiwKICAgICAgICAgICAgcmVjb3JkLmNhcGFjaXR5LCByZWNvcmQubWF4X2FicywgTUFYX0FCU19HQVRFCiAgICAgICAgKQogICAgICAgIC5pbnRvKCkpOwogICAgfQogICAgbGV0IHByb2R1Y3Rpb24gPSByZWNvcmRzLmxhc3QoKS5leHBlY3QoInJlY29yZHMgaXMgbm9uLWVtcHR5Iik7CiAgICBsZXQgc3BlZWR1cCA9IHByb2R1Y3Rpb24uc2NhbGFyX3VzIC8gcHJvZHVjdGlvbi5tbWFfdXM7CiAgICBpZiBzcGVlZHVwIDwgTUlOX1BST0RVQ1RJT05fU1BFRURVUCB7CiAgICAgICAgcmV0dXJuIEVycihmb3JtYXQhKAogICAgICAgICAgICAiV2F2ZSA2NCBzcGVlZCBnYXRlIGZhaWxlZDoge3NwZWVkdXA6LjR9eCA8IHtNSU5fUFJPRFVDVElPTl9TUEVFRFVQOi4yfXgiCiAgICAgICAgKQogICAgICAgIC5pbnRvKCkpOwogICAgfQogICAgT2soKCkpCn0K"}]')
ROOT = Path("/kaggle/working/wave75")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
ARCHIVE = Path("/kaggle/working/glcuda-t4-wave75-mma-av-results.zip")

def run(cmd, cwd=None, env=None, timeout=3600, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    print(p.stdout[-12000:], flush=True)
    print(p.stderr[-12000:], file=sys.stderr, flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, process):
    (RESULTS / name).write_text(
        process.stdout + "\n--- STDERR ---\n" + process.stderr,
        encoding="utf-8",
    )

def archive():
    if ARCHIVE.exists():
        ARCHIVE.unlink()
    with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    print("ARCHIVE", ARCHIVE, hashlib.sha256(ARCHIVE.read_bytes()).hexdigest())

try:
    shutil.rmtree(ROOT, ignore_errors=True)
    RESULTS.mkdir(parents=True)
    gpu = run(["nvidia-smi", "--query-gpu=name,compute_cap,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    gpu_line = gpu.stdout.strip().splitlines()[0]
    if "Tesla T4" not in gpu_line or "7.5" not in gpu_line:
        raise RuntimeError(f"Wave 75 requires Tesla T4 sm_75, got {gpu_line}")

    if not shutil.which("cargo"):
        installer = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", installer)
        run(["sh", installer, "-y", "--profile", "minimal"], timeout=1800)
        os.environ["PATH"] = str(Path.home() / ".cargo/bin") + os.pathsep + os.environ["PATH"]
    cargo = shutil.which("cargo") or str(Path.home() / ".cargo/bin/cargo")
    save("rust.log", run([cargo, "--version"], timeout=60))

    save("git-clone.log", run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800))
    save("git-checkout.log", run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=300))
    manifest = []
    for item in EMBEDDED:
        data = base64.b64decode(item["base64"])
        actual = hashlib.sha256(data).hexdigest()
        if actual != item["sha256"]:
            raise RuntimeError(f"embedded SHA mismatch: {item['path']}")
        path = TREE / item["path"]
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)
        manifest.append({"path": item["path"], "sha256": actual, "bytes": len(data)})
    (RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2))
    save("git-diff-check.log", run(["git", "diff", "--check"], cwd=TREE))

    ptx_path = TREE / "glcuda/src/kernels/glcuda_sm75_wave64.ptx"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    if not Path(ptxas).is_file():
        raise RuntimeError(f"ptxas not found: {ptxas}")
    assembled = run([ptxas, "-v", "-arch=sm_75", ptx_path,
                     "-o", ROOT / "wave75.cubin"], cwd=TREE, timeout=1800)
    save("ptxas-v.log", assembled)
    ptx_log = assembled.stdout + "\n" + assembled.stderr
    resources = {}
    for entry in ["gl_wave64_av_scalar_f32", "gl_wave75_av_mma4_row_f32"]:
        match = re.search(r"Compiling entry function ['\"]" + re.escape(entry) +
                          r"['\"].*?(?=Compiling entry function|\Z)", ptx_log, re.S)
        segment = match.group(0) if match else ""
        reg = re.search(r"Used (\d+) registers", segment)
        resources[entry] = {
            "found": bool(match),
            "registers": int(reg.group(1)) if reg else None,
            "spill_stores": max([int(x) for x in re.findall(r"(\d+) bytes spill stores", segment)] or [0]),
            "spill_loads": max([int(x) for x in re.findall(r"(\d+) bytes spill loads", segment)] or [0]),
            "stack": max([int(x) for x in re.findall(r"(\d+) bytes stack frame", segment)] or [0]),
        }
    (RESULTS / "resources.json").write_text(json.dumps(resources, indent=2))
    if not all(x["found"] for x in resources.values()):
        raise RuntimeError(f"missing ptxas entry: {resources}")
    if any(x["spill_stores"] or x["spill_loads"] or x["stack"] for x in resources.values()):
        raise RuntimeError(f"resource gate failed: {resources}")

    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave64_mma_av", "--locked"], cwd=TREE, timeout=7200)
    save("cargo-build.log", build)
    measured = run([TREE / "target/release/examples/wave64_mma_av"], cwd=TREE,
                   env={"GLCUDA_JIT_VERBOSE": "1", "CUDA_VISIBLE_DEVICES": "0"},
                   timeout=7200, check=False)
    save("wave75-direct.log", measured)
    result_line = next((x for x in measured.stdout.splitlines() if x.startswith("[wave64-av] ")), "")
    resource_line = next((x for x in measured.stdout.splitlines() if x.startswith("[wave64-resource] ")), "")
    result = {
        "base_revision": BASE_REV,
        "gpu": gpu_line,
        "ptxas": resources,
        "driver": json.loads(resource_line.split("] ", 1)[1]) if resource_line else None,
        "direct": json.loads(result_line.split("] ", 1)[1]) if result_line else None,
        "returncode": measured.returncode,
    }
    (RESULTS / "result.json").write_text(json.dumps(result, indent=2))
    if measured.returncode:
        raise RuntimeError("Wave 75 direct gate failed")
    (RESULTS / "PASS.json").write_text(json.dumps({"status": "feasibility_pass"}, indent=2))
    archive()
except Exception:
    RESULTS.mkdir(parents=True, exist_ok=True)
    (RESULTS / "FAILED.txt").write_text(traceback.format_exc())
    archive()
    raise
